# Create a hold-out set for final testing

Author: Pete King

This notebook creates a separate testing dataset to hold in reserve for a final evaluation of our selected model's ability to generalize to unseen data.

We want to rigorously test our final model against a period of dynamic real-world volatility.  For example, beginning in October 2024 would capture an initial period of relatively low volatility, followed by the highly volatility period after President Trump announced reciprocal tarrifs in April 2025.

In [1]:
#123456789012345678901234567890123456789012345678901234567890123456789012345678
import re

import pandas as pd

import data_prep as dp

In [2]:
# Select the data file and start date for the testing set
DATA_FILENAME = 'etf_with_return.csv'
TESTING_START_DATE = '2024-10-01'
# Depending on how we calculate our target variables, we may need a buffer
# between training and testing sets.
BUFFER_SIZE = 0

In [3]:
# Import feature dataset with labels
df = pd.read_csv(
    DATA_FILENAME,
    index_col='date',
    parse_dates=True
)

In [4]:
# Create separate training/validation and testing data files
train_val_df, test_df = dp.time_series_split(
    df, TESTING_START_DATE, BUFFER_SIZE
)
train_val_df

,BIL,BND,GLD,HYG,IEF,IWM,LQD,QQQ,SPY,TIP,...,XLB_return,XLE_return,XLF_return,XLI_return,XLK_return,XLP_return,XLRE_return,XLU_return,XLV_return,XLY_return
date,,,,,,,,,,,,,,,,,,,,,
1993-01-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.241400,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.413815,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.465536,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.724159,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.827616,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-09-24,86.178871,71.128273,246.070007,73.447990,93.066467,216.896286,106.184669,482.143372,562.774902,106.529709,...,0.013563,-0.002472,-0.008398,0.007158,0.006821,-0.002410,0.000000,-0.007121,-0.002010,0.008720
2024-09-25,86.197647,70.863937,245.729996,73.365479,92.688400,214.250748,105.557747,482.590363,561.533630,106.125992,...,-0.006706,-0.019767,-0.006010,-0.004692,0.002858,-0.002537,-0.004245,0.005252,-0.009782,-0.003599
2024-09-26,86.197647,70.863937,246.979996,73.484673,92.622238,215.392410,105.557747,486.216095,563.760010,105.991409,...,0.020499,-0.019814,0.005122,0.004989,0.013157,0.002295,-0.009899,-0.006883,0.003337,0.004248


In [5]:
test_df

,BIL,BND,GLD,HYG,IEF,IWM,LQD,QQQ,SPY,TIP,...,XLB_return,XLE_return,XLF_return,XLI_return,XLK_return,XLP_return,XLRE_return,XLU_return,XLV_return,XLY_return
date,,,,,,,,,,,,,,,,,,,,,
2024-10-01,86.255020,71.120422,245.610001,73.576149,93.073112,214.447586,106.224892,478.070618,560.134827,106.532005,...,-0.002493,0.022524,-0.005310,0.000517,-0.024664,-0.003379,-0.006062,0.008014,-0.005273,-0.004051
2024-10-02,86.264442,70.987831,245.660004,73.585373,92.807693,214.152344,105.990097,478.746094,560.371216,106.435760,...,-0.003856,0.010193,0.001109,-0.001847,0.007373,-0.008498,-0.003610,0.000491,-0.002025,-0.008505
2024-10-03,86.283318,70.675392,245.490005,73.410316,92.333755,212.715424,105.304535,478.388489,559.346802,105.983521,...,-0.011130,0.017483,-0.005332,-0.005114,0.004138,-0.010048,-0.009767,-0.000368,-0.009066,-0.011641
2024-10-04,86.302177,70.211418,245.000000,73.327423,91.442741,215.687683,104.731674,484.080383,564.429810,105.261826,...,0.004845,0.009272,0.016789,0.006961,0.011069,0.003320,-0.006642,-0.001598,0.000792,0.013156
2024-10-07,86.321053,70.003090,244.169998,73.014221,91.110985,213.994843,104.196365,478.895111,559.327087,105.146355,...,-0.002946,0.003535,-0.012343,-0.002438,-0.006904,-0.009992,-0.007380,-0.023267,-0.004428,-0.015664
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-13,91.510002,73.550003,460.839996,79.199997,95.589996,246.152130,108.169998,593.719971,662.289978,110.709999,...,-0.009912,0.003298,0.001228,-0.003577,-0.007574,0.005799,0.002607,0.009844,-0.002467,-0.005936
2026-03-16,91.510002,73.830002,460.429993,79.449997,96.019997,248.477997,108.690002,600.380005,669.030029,111.059998,...,0.004260,0.003460,0.008351,0.008527,0.014370,0.002828,0.007780,0.006368,0.008112,0.012015
2026-03-17,91.519997,73.980003,459.269989,79.809998,96.190002,250.050003,109.300003,603.309998,670.789978,111.449997,...,0.002426,0.010480,0.005260,0.002646,0.005461,-0.003300,0.003283,-0.002754,-0.009114,0.008697


In [6]:
# Save segmented datasets for further analysis
train_val_df.to_csv('train_val_data.csv')
test_df.to_csv('test_data.csv')